In [9]:
import psycopg2
import numpy as np
from langchain_openai import OpenAIEmbeddings

In [24]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv("OPENAI_API_KEY") is not None)

True


In [19]:
texts = [
    "Python is a high-level programming language commonly used for data science, machine learning, and web development.",

    "PostgreSQL is a relational database management system that supports SQL and can store structured application data.",

    "Vector databases store numerical embeddings and allow applications to perform semantic similarity searches.",

    "RAG systems retrieve relevant documents from a knowledge base and provide them to a language model as context.",

    "FastAPI is a Python web framework commonly used to build APIs and backend services.",

    "NumPy is a Python library for numerical computing that provides multidimensional arrays and mathematical operations.",

    "PyTorch is a machine learning framework commonly used for deep learning and neural network development.",

    "Git is a version control system used to track changes to source code and collaborate with other developers.",
]

In [27]:
embeddings = OpenAIEmbeddings()

embeddings_list = []

for text in texts:
    embeddings_list.append(embeddings.embed_query(text))
print(len(embeddings_list[0]))

1536


In [30]:
connection =  psycopg2.connect("dbname=vectortutorialdb user=postgres password=postgres")
cursor = connection.cursor()

for i in range(len(embeddings_list)):
  embedding = embeddings_list[i]
  content = texts[i]
  cursor.execute("INSERT INTO items (content, embedding) VALUES (%s, %s)", (content, embedding))

connection.commit()

cursor.close()
connection.close()

In [34]:
new_text = "How can I give an AI model relevant information from a collection of stored documents?"
new_embedding = embeddings.embed_query(new_text)

connection = psycopg2.connect("dbname=vectortutorialdb user=postgres password=postgres")
cursor = connection.cursor()
cursor.execute("""SELECT id, content
               FROM items
               ORDER BY embedding <-> %s::vector
               LIMIT 3""", (new_embedding,))

In [35]:
results = cursor.fetchall()
for row in results:
  print(row)

(4, 'RAG systems retrieve relevant documents from a knowledge base and provide them to a language model as context.')
(3, 'Vector databases store numerical embeddings and allow applications to perform semantic similarity searches.')
(7, 'PyTorch is a machine learning framework commonly used for deep learning and neural network development.')


In [ ]:
connection.close()